# Customer Churn Prediction — Baseline Notebook

**Purpose:** educational baseline for the RabTech Academy Task 2 deliverable.

The supplied dataset has only 12 rows. This is far too small for dependable generalization or fairness conclusions. Treat all results as a demonstration of workflow, not as evidence that the model is ready for real customers.

Target assumption: `churned=1` means churn and `churned=0` means no churn. Confirm this with the data provider.


In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

df = pd.read_csv("customer-churn-training.csv")
print("Shape:", df.shape)
display(df.head())
print("\\nData types:")
print(df.dtypes)
print("\\nMissing values:")
print(df.isna().sum())
print("\\nExact duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())
print("\\nTarget counts:")
print(df["churned"].value_counts(dropna=False))
print("\\nPlan counts:")
print(df["plan_type"].value_counts(dropna=False))


Shape: (12, 7)


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0


\nData types:
customer_id            str
tenure_months        int64
support_tickets      int64
monthly_spend_inr    int64
last_login_days      int64
plan_type              str
churned              int64
dtype: object
\nMissing values:
customer_id          0
tenure_months        0
support_tickets      0
monthly_spend_inr    0
last_login_days      0
plan_type            0
churned              0
dtype: int64
\nExact duplicate rows: 0
Duplicate customer IDs: 0
\nTarget counts:
churned
0    7
1    5
Name: count, dtype: int64
\nPlan counts:
plan_type
Basic       5
Standard    4
Pro         3
Name: count, dtype: int64


## 1. Define features and target

`customer_id` is an identifier and is intentionally excluded from model features. The prediction features must be available before the prediction date; verify this with the data owner to avoid leakage.


In [13]:
target = "churned"
feature_cols = [
    "tenure_months",
    "support_tickets",
    "monthly_spend_inr",
    "last_login_days",
    "plan_type",
]
X = df[feature_cols].copy()
y = df[target].astype(int)

numeric_features = [
    "tenure_months", "support_tickets",
    "monthly_spend_inr", "last_login_days"
]
categorical_features = ["plan_type"]

print("Rows:", len(df))
print("Churn labels:", y.value_counts().to_dict())


Rows: 12
Churn labels: {0: 7, 1: 5}


## 2. Train/test split and preprocessing

A stratified split is attempted to preserve the label ratio. With only 12 records, the test set is tiny, so each prediction has a large effect on metrics. Preprocessing is fitted only on the training split by keeping it inside a scikit-learn pipeline.


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train label counts:", y_train.value_counts().to_dict())
print("Test label counts:", y_test.value_counts().to_dict())


Training rows: 9
Test rows: 3
Train label counts: {0: 5, 1: 4}
Test label counts: {0: 2, 1: 1}


## 3. Fit simple baselines

Compare a majority-class DummyClassifier with Logistic Regression. The goal is to demonstrate a reproducible baseline, not to tune a model on this tiny sample.


In [15]:
models = {
    "Majority-class baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
}

results = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    pred = model.predict(X_test)
    row = {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    }
    if hasattr(model, "predict_proba") and len(np.unique(y_test)) == 2:
        row["roc_auc"] = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    else:
        row["roc_auc"] = np.nan
    results.append(row)

results_df = pd.DataFrame(results)
display(results_df)


,model,accuracy,precision,recall,f1,roc_auc
0,Majority-class baseline,0.666667,0.0,0.0,0.0,0.5
1,Logistic Regression,1.000000,1.0,1.0,1.0,1.0


## 4. Inspect errors

The confusion matrix uses label order `[0, 1]` = `[not churned, churned]`. Review false positives and false negatives carefully. Because the test set contains only three records, the error counts and metrics are extremely unstable.


In [16]:
best_model = fitted_models["Logistic Regression"]
pred = best_model.predict(X_test)

print("Confusion matrix (rows=true, columns=predicted; labels 0 then 1):")
print(confusion_matrix(y_test, pred, labels=[0, 1]))
print("\\nClassification report:")
print(classification_report(y_test, pred, labels=[0, 1], zero_division=0))


Confusion matrix (rows=true, columns=predicted; labels 0 then 1):
[[2 0]
 [0 1]]
\nClassification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



## 5. Conclusion and limitations

- The dataset has only 12 rows, no missing values in the supplied file, no exact duplicate rows, and no repeated customer IDs.
- The target counts are 7 non-churn and 5 churn records.
- Do not claim production readiness, reliable generalization, or fairness based on this sample.
- Confirm dataset provenance, licence/permission, target meaning, feature definitions, and feature timestamps with the provider.
- For a serious evaluation, obtain a larger representative dataset, use an appropriate holdout or cross-validation strategy, examine calibration and errors, and assess fairness where lawful and feasible.
